### Setup

In [1]:
import pandas as pd
import os

from neo4j import GraphDatabase
from decimal import Decimal
from dotenv import load_dotenv

from shapely import wkb
from pyproj import CRS, Transformer


load_dotenv("../.env")

NODES_DIR = "../data/nodes/"

In [ ]:
class Config:
    def __init__(self, mode="LOCAL", DATABASE='neo4j'):
        mode = mode.upper()

        if mode == "LOCAL":
            self.URI = os.getenv("NEO4J_URI_LOCAL")
            self.USER = os.getenv("NEO4J_USER_LOCAL")
            self.PASSWORD = os.getenv("NEO4J_PASSWORD_LOCAL")
        elif mode == "GROUP":
            self.URI = os.getenv("NEO4J_GROUP_URI")
            self.USER = os.getenv("NEO4J_GROUP_USER")
            self.PASSWORD = os.getenv("NEO4J_GROUP_PASSWORD")
        else:
            raise ValueError("Mode must be 'LOCAL' or 'GROUP'.")
        self.DATABASE = DATABASE


config = Config(mode="LOCAL",DATABASE = "dse203test")
driver = GraphDatabase.driver(config.URI, auth=(config.USER, config.PASSWORD))

with driver.session(database=config.DATABASE) as session:
    session.run("MATCH (n) RETURN n LIMIT 1")
    print(f"Connection successful")

def query_neo4j(cypher_query: str, parameters: dict = None):
    with driver.session(database=config.DATABASE) as session:
        result = session.run(cypher_query, parameters)
        return result.data()

Connection successful


In [3]:
def delete_nodes(label: str):
    label_exists = query_neo4j(f"CALL db.labels() YIELD label WHERE label = '{label}' RETURN count(*) > 0 AS exists")[0]['exists']
    if label_exists:
        deleted_count = query_neo4j(f"MATCH (n:`{label}`) DETACH DELETE n RETURN count(n) AS deleted_count")[0]["deleted_count"]
        print(f"Deleted {deleted_count} existing {label} nodes.")
    else:
        print(f"No {label} nodes found.")

def clean_data(data: dict):
    cleaned_data = []
    for row in data:
        cleaned_row = {k: float(v) if isinstance(v, Decimal) else v for k, v in row.items()}
        if cleaned_row:
            cleaned_data.append(cleaned_row)
    return cleaned_data

def create_nodes(label: str, data: dict):
    data = clean_data(data)
    create_query = f"""
        UNWIND $data AS row
        CREATE (n:`{label}`)
        SET n = row
        RETURN count(n) AS created
    """
    result = query_neo4j(create_query, {"data": data})
    created_count = result[0]['created']
    print(f"Created {created_count} new {label} nodes.")

def drop_indexes(label):
    # Step 1: get all indexes for this label
    get_indexes_query = """
        SHOW INDEXES YIELD name, labelsOrTypes
        WHERE $label IN labelsOrTypes
        RETURN name
    """
    results = query_neo4j(get_indexes_query, {"label": label})

    # Step 2: drop each index
    for row in results:
        name = row["name"]
        drop_query = f"DROP INDEX {name}"
        query_neo4j(drop_query)
    print(f"Dropped indexes for label `{label}`.")


def create_index_on_id(label: str):
    index_name = f"{label.lower()}_id_index"
    drop_index_query = f"DROP INDEX {index_name} IF EXISTS"
    create_index_query = f"CREATE INDEX {index_name} FOR (n:`{label}`) ON (n.id)"
    query_neo4j(drop_index_query)
    query_neo4j(create_index_query)
    print(f"Created index `{index_name}` on {label}(id).")

def load_nodes(label, filename: str):
    print(f"\nLoading {label} nodes...")
    # Detect file type and use appropriate reader
    if filename.endswith('.csv'):
        df = pd.read_csv(NODES_DIR + filename)
    else:
        df = pd.read_json(NODES_DIR + filename)
    data = df.to_dict(orient="records")
    drop_indexes(label)
    create_index_on_id(label)
    delete_nodes(label)
    create_nodes(label, data)


def create_wkt_layer(tx, layer_name, wkt_property):
    query = """
    CALL spatial.addWKTLayer($layerName, $wktProperty)
    """
    tx.run(query, layerName=layer_name, wktProperty=wkt_property)
    print(f"WKT layer '{layer_name}' created or already exists.")

### Load Nodes From JSON

In [4]:
node_files = os.listdir(NODES_DIR)

# Exclude intermediate sector files - these are loaded separately
exclude_files = {'business_subsector.json', 'sectors.json', 'subsectors.json'}
node_files = [f for f in node_files if f not in exclude_files]

labels = {
    "".join(c.capitalize() for c in filename.split(".")[0].split("_")): filename 
    for filename in node_files
}
labels

{'BlockGroup': 'block_group.json',
 'Business': 'business.json',
 'BusinessLocation': 'business_location.json',
 'City': 'city.json',
 'Community': 'community.json',
 'County': 'county.json',
 'State': 'state.json',
 'Zipcode': 'zipcode.json',
 'ZoneLocation': 'zone_location.json',
 'ZoneType': 'zone_type.json'}

In [8]:
for label, file in labels.items():
    load_nodes(label, file)


Loading BlockGroup nodes...
Dropped indexes for label `BlockGroup`.
Created index `blockgroup_id_index` on BlockGroup(id).
Deleted 0 existing BlockGroup nodes.
Created 2085 new BlockGroup nodes.

Loading Business nodes...
Dropped indexes for label `Business`.
Created index `business_id_index` on Business(id).
Deleted 0 existing Business nodes.
Created 32010 new Business nodes.

Loading BusinessLocation nodes...
Dropped indexes for label `BusinessLocation`.
Created index `businesslocation_id_index` on BusinessLocation(id).
Deleted 0 existing BusinessLocation nodes.
Created 39593 new BusinessLocation nodes.

Loading City nodes...
Dropped indexes for label `City`.
Created index `city_id_index` on City(id).
Deleted 0 existing City nodes.
Created 52 new City nodes.

Loading Community nodes...
Dropped indexes for label `Community`.
Created index `community_id_index` on Community(id).
Deleted 0 existing Community nodes.
Created 229 new Community nodes.

Loading County nodes...
Dropped indexe

### Create And Update Nodes Using Cypher

In [9]:
BATCH_SIZE = 5000 
SOURCE_CRS = CRS.from_epsg(2230)
TARGET_CRS = CRS.from_epsg(4326)
WKT_PROPERTY = "geom_wkt"
LAYER_NAME = "WKTLayer"


In [12]:
#set business location

with driver.session(database=config.DATABASE) as session:
    match_query = 'MATCH (n:BusinessLocation)'
    query_constraint = 'WHERE n.latitude IS NOT NULL AND n.longitude IS NOT NULL '
    update_query = 'SET n.location = point({ latitude: toFloat(n.latitude), longitude: toFloat(n.longitude)}) '
     
    result = session.run(match_query + query_constraint+ update_query)

#create spatial index
with driver.session(database=config.DATABASE) as session:
    query = "CREATE POINT INDEX idx_location IF NOT EXISTS FOR (n:BusinessLocation) ON (n.location)"

    result = session.run(query)


In [14]:
def setup_wkt_spatial_layer(driver, label):
    print(f"Processing spatial layer for: {label}")

    cypher = f"""
    MATCH (n:{label})
    WHERE n.centroid_wkt IS NOT NULL AND n.centroid_wkt STARTS WITH 'POINT'
    WITH n, split(replace(replace(n.centroid_wkt,'POINT(',' '),')',''), ' ') AS parts
    WITH n, toFloat(parts[1]) AS lon, toFloat(parts[2]) AS lat
    WHERE lon IS NOT NULL AND lat IS NOT NULL
    SET n.location = point({{longitude: lon, latitude: lat, crs:'WGS-84'}})
    """
    with driver.session(database=config.DATABASE) as session:
        result = session.run(cypher)
        summary = result.consume()
        print(f"  - Updated {summary.counters.properties_set} nodes with spatial points.")

    with driver.session(database=config.DATABASE) as session:
        index_name = f"idx_{label.lower()}_location"
        query = f"CREATE POINT INDEX {index_name} IF NOT EXISTS FOR (n:{label}) ON (n.location)"

        session.run(query)
        print(f"  - Index '{index_name}' created/verified.")


# Run for City
setup_wkt_spatial_layer(driver, "City")

# Run for BlockGroup
setup_wkt_spatial_layer(driver, "BlockGroup")


Processing spatial layer for: City
  - Updated 52 nodes with spatial points.
  - Index 'idx_city_location' created/verified.
Processing spatial layer for: BlockGroup
  - Updated 2085 nodes with spatial points.
  - Index 'idx_blockgroup_location' created/verified.


### Load Sectors and Subsectors from Business Locations

In [17]:
import json
from pathlib import Path

# Load business locations with sector data
SECTORS_DATA_PATH = Path("../data/ca_businesses_with_ai_franchise_sectors.json")
with open(SECTORS_DATA_PATH, 'r', encoding='utf-8') as f:
    business_with_sectors = json.load(f)

df_sectors = pd.DataFrame(business_with_sectors)
print(f"Loaded {len(df_sectors)} business locations with sector data")

Loaded 39593 business locations with sector data


In [18]:
# Extract unique sectors
sectors = sorted(df_sectors['category_sector'].dropna().unique().tolist())
sectors_payload = [{'name': sector} for sector in sectors]
print(f"Found {len(sectors)} unique sectors")

# Extract unique subsectors with their parent sectors
subsectors_df = (
    df_sectors[['category_sector', 'category_subsector']]
    .dropna()
    .drop_duplicates()
    .rename(columns={'category_sector': 'sector', 'category_subsector': 'subsector'})
)
subsectors_payload = subsectors_df.to_dict(orient='records')
print(f"Found {len(subsectors_payload)} unique subsectors")

Found 12 unique sectors
Found 45 unique subsectors


In [19]:
# Create constraints for data integrity
with driver.session(database=config.DATABASE) as session:
    session.run("CREATE CONSTRAINT sector_name IF NOT EXISTS FOR (s:Sector) REQUIRE s.name IS UNIQUE")
    session.run("CREATE CONSTRAINT subsector_name IF NOT EXISTS FOR (s:Subsector) REQUIRE s.name IS UNIQUE")
    print("Created constraints for Sector and Subsector")

Created constraints for Sector and Subsector


In [20]:
# Create Sector nodes
create_sectors_query = """
UNWIND $rows AS row
MERGE (:Sector {name: row.name})
"""
with driver.session(database=config.DATABASE) as session:
    result = session.run(create_sectors_query, {'rows': sectors_payload})
    summary = result.consume()
    print(f"Created {summary.counters.nodes_created} Sector nodes")

Created 12 Sector nodes


In [21]:
# Create Subsector nodes and link to Sectors
create_subsectors_query = """
UNWIND $rows AS row
MATCH (sec:Sector {name: row.sector})
MERGE (sub:Subsector {name: row.subsector})
MERGE (sub)-[:IS_TYPE]->(sec)
"""
with driver.session(database=config.DATABASE) as session:
    result = session.run(create_subsectors_query, {'rows': subsectors_payload})
    summary = result.consume()
    print(f"Created {summary.counters.nodes_created} Subsector nodes")
    print(f"Created {summary.counters.relationships_created} IS_TYPE relationships")

Created 45 Subsector nodes
Created 45 IS_TYPE relationships


In [22]:
# Link BusinessLocation nodes to Subsectors
# Process in batches to handle large dataset
business_payload = df_sectors[['id', 'category_subsector']].dropna().to_dict(orient='records')

link_business_query = """
UNWIND $rows AS row
MATCH (b:BusinessLocation {id: row.id})
MATCH (sub:Subsector {name: row.category_subsector})
MERGE (b)-[:IN_SUBSECTOR]->(sub)
"""

batch_size = 5000
total_relationships = 0

with driver.session(database=config.DATABASE) as session:
    for i in range(0, len(business_payload), batch_size):
        batch = business_payload[i:i+batch_size]
        result = session.run(link_business_query, {'rows': batch})
        summary = result.consume()
        total_relationships += summary.counters.relationships_created
        if (i // batch_size) % 10 == 0:
            print(f"Processed {i + len(batch)}/{len(business_payload)} business locations...")

print(f"Created {total_relationships} IN_SUBSECTOR relationships")

Processed 5000/39593 business locations...
Created 39593 IN_SUBSECTOR relationships
